# 03 Scoring and H3 Export

## Scoring Logic

This notebook inspects the generated H3 scoring output used by the Web app. The composite score is `0.55 * baseline_score + 0.45 * trackA_score`. The baseline score represents everyday 15-minute access using food, shopping, health, education, transit, and green/public facilities. The Track A score represents healthy-lifestyle access using sport, green, cycling-support, fresh-food, and air-quality signals.

The 55/45 split keeps the universal 15-minute-city baseline slightly stronger than the chosen track. This follows the brief: every track must first satisfy the baseline layer, then add a second story-specific layer. Track A is still heavily weighted because the project question is not general convenience but healthy lifestyle and sport access. Raw category signals are converted to empirical percentiles before compositing so that central areas with extreme POI density do not all collapse to the same 100% result.

## H3 Aggregation and Export

The final Web map uses H3 resolution 8 polygons because the brief asks for H3 r8 output. The companion `outputs/sh15_trackA_500m_grid_proxy.geojson` keeps a 500 m grid proxy linked to the H3 result, while this notebook checks the final H3 scoring layer that drives the app. The exports are `outputs/sh15_trackA_h3_r8_scored.geojson`, `webapp/data/sh15_trackA_h3_r8_scored.geojson`, and `outputs/sh15_trackA_top10_h3.csv`.

## Interpretation

Scores are empirical percentiles, not raw POI counts. This avoids over-rewarding dense central cells where many categories would otherwise saturate at 100%. The Web app keeps the mode-specific baseline fields for walk, bike, transit, and car, and lets the user change recommendation weights and district filters. The Top10 output should be read as a current-scenario recommendation list rather than a universal ranking.

## Export Use

The generated GeoJSON feeds the interactive map. The CSV output supports reporting, while the Web app can export the current Top10, current weights, and a selected-H3 report for presentation or written analysis.


In [1]:
from pathlib import Path
import csv
import json
import statistics

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
geojson = json.loads((ROOT / 'outputs' / 'sh15_trackA_h3_r8_scored.geojson').read_text(encoding='utf-8'))
grid = json.loads((ROOT / 'outputs' / 'sh15_trackA_500m_grid_proxy.geojson').read_text(encoding='utf-8'))
features = geojson['features']
scores = [float(feature['properties']['composite_score']) for feature in features]
print('features:', len(features))
print('500m grid proxy features:', len(grid['features']))
print('min:', min(scores), 'median:', statistics.median(scores), 'max:', max(scores))
print('mean:', statistics.mean(scores))


features: 11344
500m grid proxy features: 11344
min: 0.0023 median: 0.497 max: 0.9968
mean: 0.49995574753173483


In [2]:
top_path = ROOT / 'outputs' / 'sh15_trackA_top10_h3.csv'
with top_path.open('r', encoding='utf-8-sig', newline='') as f:
    for row in csv.DictReader(f):
        print(row)


{'rank': '1', 'h3_r8': '883099598dfffff', 'district': '徐汇区', 'composite_score': '0.9968', 'baseline_score': '0.9994', 'trackA_score': '0.9935', 'mean_aqi': '64.69', 'sport_count': '190', 'green_count': '100', 'education_count': '8', 'transit_count': '108'}
{'rank': '2', 'h3_r8': '88309959adfffff', 'district': '徐汇区', 'composite_score': '0.9967', 'baseline_score': '0.9944', 'trackA_score': '0.9996', 'mean_aqi': '64.69', 'sport_count': '81', 'green_count': '95', 'education_count': '0', 'transit_count': '126'}
{'rank': '3', 'h3_r8': '88309959e3fffff', 'district': '徐汇区', 'composite_score': '0.9965', 'baseline_score': '0.9992', 'trackA_score': '0.9933', 'mean_aqi': '64.69', 'sport_count': '83', 'green_count': '105', 'education_count': '9', 'transit_count': '71'}
{'rank': '4', 'h3_r8': '88309959a9fffff', 'district': '徐汇区', 'composite_score': '0.9962', 'baseline_score': '0.9976', 'trackA_score': '0.9946', 'mean_aqi': '64.69', 'sport_count': '42', 'green_count': '24', 'education_count': '2', 't